In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModel, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [ ]:
# Split based on which side words were spoken to
df = pd.read_csv('extracted_labelled.csv')
df = df[df['total_words'] > 200]
df['utterance_list'] = df['all_utterances'].str.split('|||', regex=False).apply(lambda x: [s.strip() for s in x])

side0_col = []
side1_col = []

for i in df['utterance_list']:
    arr_side0 = []
    arr_side1 = []

    for u in i:
        if u.startswith('[SIDE0]'):
            arr_side0.append(u[7:])
        elif u.startswith('[SIDE1]'):
            arr_side1.append(u[7:])

    side0_col.append(arr_side0)
    side1_col.append(arr_side1)

df['words_to_side0'] = side0_col
df['words_to_side1'] = side1_col

df.head()

,case_id,year,court,justice_id,label,win_side,all_utterances,total_words,total_utterances,words_to_side0,...,word_ratio_0_to_1,questions_to_side0,questions_to_side1,question_ratio_0_to_1,interruptions,issue_area,decision_direction,maj_votes,min_votes,utterance_list
0,1955_71,1955,Warren Court,j__earl_warren,0,0.0,"[UNKNOWN] Number 71, Lonnie Affronti versus Un...",230,17,"[ -- one view or the other., Mr. Lindsay, man...",...,2.235294,2,5,0.333333,0,9.0,1.0,9,0,"[[UNKNOWN] Number 71, Lonnie Affronti versus U..."
2,1955_71,1955,Warren Court,j__felix_frankfurter,0,0.0,[SIDE1] This -- is it not necessary for you to...,1209,49,"[ Is that a case like this?, I don't -- I don...",...,4.062762,5,2,1.666667,0,9.0,1.0,9,0,[[SIDE1] This -- is it not necessary for you t...
5,1955_410,1955,Warren Court,j__stanley_reed,0,1.0,[SIDE1] Is -- is there another company in the ...,202,14,[ But you have to -- you have to determine wha...,...,0.109290,1,5,0.166667,0,8.0,1.0,7,2,[[SIDE1] Is -- is there another company in the...
6,1955_410,1955,Warren Court,j__felix_frankfurter,1,1.0,"[SIDE1] Mr. Westwood, before you sit down I ta...",544,19,[ Is -- is the suggestion that no question of ...,...,0.159574,4,1,2.000000,0,8.0,1.0,7,2,"[[SIDE1] Mr. Westwood, before you sit down I t..."
7,1955_351,1955,Warren Court,j__earl_warren,1,1.0,"[UNKNOWN] Number 351, R.V. Archawski et al. ve...",285,15,[],...,0.000000,0,5,0.000000,0,8.0,2.0,9,0,"[[UNKNOWN] Number 351, R.V. Archawski et al. v..."


In [13]:
df = df[['justice_id', 'words_to_side0', 'words_to_side1', 'label']]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("lexlms/legal-longformer-base")
model = AutoModel.from_pretrained("lexlms/legal-longformer-base")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
X = df[['words_to_side0', 'words_to_side1']]
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=12345)

In [ ]:
#Tokenize
def tokenize(X, y):
    texts_a = X['words_to_side0'].apply(lambda x: " ".join(x))
    texts_b = X['words_to_side1'].apply(lambda x: " ".join(x))

    encodings = tokenizer(list(texts_a), list(texts_b), padding="max_length", truncation=True, max_length=8192, return_tensors="pt")
    return encodings, torch.tensor(y.values, dtype=torch.long)

train_encodings, train_labels = tokenize(X_train, y_train)
test_encodings, test_labels = tokenize(X_test, y_test)

In [ ]:
class SuperScotus(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, x):
        return {
            "input_ids": self.encodings["input_ids"][x],
            "attention_mask": self.encodings["attention_mask"][x],
            "token_type_ids": self.encodings["token_type_ids"][x],
            "labels": self.labels[x]
        }

train_loader = DataLoader(SuperScotus(train_encodings, train_labels), batch_size=16, shuffle=True)
test_loader = DataLoader(SuperScotus(test_encodings, test_labels), batch_size=16, shuffle=False)

In [35]:
class ScotusLegalBERT(nn.Module):
    def __init__(self, model, num_classes, dropout= 0.1):
        super().__init__()
        self.bert = model
        hidden_size = model.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled = outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits

    
classifier = ScotusLegalBERT(model, num_classes=2).to(device)

In [ ]:
#config
EPOCHS = 5
LR = 2e-5
WARMUP  = 0.1

num_train_steps = len(train_loader) * EPOCHS
warmup_steps    = int(WARMUP * num_train_steps)

optimizer = AdamW(classifier.parameters(), lr=LR, weight_decay=0.01)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=num_train_steps
)
loss_fn = nn.CrossEntropyLoss().to(device)


In [ ]:
# Training loop

def run_epoch(model, loader, optimizer=None, scheduler=None, training=True):
    model.train() if training else model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels         = batch['labels'].to(device)

            logits = model(input_ids, attention_mask, token_type_ids)
            loss   = loss_fn(logits, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, acc, f1


for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_f1 = run_epoch(classifier, train_loader, optimizer, scheduler, training=True)
    print(f"Epoch {epoch}/{EPOCHS} | Train loss: {tr_loss:.4f}  acc: {tr_acc:.4f}  f1: {tr_f1:.4f}")

torch.save(classifier.state_dict(), 'best_legal_bert.pt')
print("Model saved.")

In [ ]:
# Evaluate on the held-out test set

classifier.load_state_dict(torch.load('best_legal_bert.pt', map_location=device))

classifier.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        preds = torch.argmax(
            classifier(input_ids, attention_mask, token_type_ids), dim=1
        ).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['labels'].tolist())

test_acc = accuracy_score(all_labels, all_preds)
test_f1  = f1_score(all_labels, all_preds, average='weighted')
print(f"Test Accuracy: {test_acc:.4f}  |  Weighted F1: {test_f1:.4f}\n")
print(classification_report(all_labels, all_preds, target_names=['Side 0 wins', 'Side 1 wins']))